# Marginalized Chebyshev continuum: shrinkage and additive jitter

This small, reproducible **continuum-only** example uses `model_flux = 1`. The truth has degree 2; analysis uses the default degree 4 with independent Gaussian coefficient priors. No stellar spectrum, external data, SpecFit, optimization or NumPyro is needed.

We examine conditional continuum recovery, the uncertainty of unnecessary coefficients, and fully normalized marginal likelihoods for continuum prior scale and additive jitter. All scales below are illustrative flux-unit choices, not package defaults. Run from inside the jaxstar checkout using a kernel with JAX, NumPy and Matplotlib. Generated figures stay under ignored `benchmark-results/continuum-demo/`.


In [ ]:
from pathlib import Path
import os
import sys

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "src/jaxstar/specfit/continuum.py").is_file()), None)
if repo_root is None:
    raise RuntimeError("Run from inside the jaxstar checkout")
work_dir = repo_root / "benchmark-results/continuum-demo"
work_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(work_dir / "matplotlib"))
sys.path.insert(0, str(repo_root / "src"))

import jax
import jax.numpy as jnp
import numpy as np
# Explicit tutorial precision choice; the library does not enable x64.
jax.config.update("jax_enable_x64", True)
%matplotlib inline
import matplotlib.pyplot as plt
from jaxstar.specfit import (Observation, chebyshev_basis, evaluate_continuum,
                            continuum_posterior, marginalized_continuum_log_likelihood)
print("JAX:", jax.__version__, "backend:", jax.default_backend())


## Generate a degree-2 continuum and noisy observation

At each wavelength, the simulated noise variance is `sigma_obs**2 + jitter_true**2`. Jitter is additive **absolute** noise in the same flux units as `Observation.uncertainty`, not a fraction of model flux. Observation stores only the measured uncertainty; jitter is passed separately to the likelihood/posterior. Two short intervals are masked, and their flux/error values deliberately contain NaN/Inf.


In [ ]:
rng = np.random.default_rng(20261004)
wavelength = np.linspace(5000., 5020., 120)
model_flux = jnp.ones(wavelength.shape)
true_coefficients = np.array([1.04, .04, -.025, 0., 0.])
basis = chebyshev_basis(wavelength, degree=4)
true_continuum = np.asarray(evaluate_continuum(basis[..., :3], true_coefficients[:3]))
sigma_obs, jitter_true = .012, .008
noise_std = np.hypot(sigma_obs, jitter_true)
measured = true_continuum + rng.normal(0., noise_std, size=wavelength.shape)
mask = np.zeros(wavelength.shape, dtype=bool)
mask[18:24], mask[85:92] = True, True
flux, uncertainty = measured.copy(), np.full(wavelength.shape, sigma_obs)
flux[mask], uncertainty[mask] = np.nan, np.inf
obs = Observation(wavelength, flux, uncertainty, mask)
sigma_constant = .1
candidate_scales = (.002, .03, .3)
reference_scale = .03
print("Truth degree: 2; analysis degree: 4; usable pixels:", obs.valid.sum())


## A. Conditional continuum recovery

For each prior scale, recover the coefficient posterior analytically. The band is the **conditional latent-continuum** mean ±2 standard deviations for `sigma_continuum=0.03`, with jitter fixed at its injected value. It excludes future observational noise and uncertainty in the hyperparameters. Very strong shrinkage can suppress real slope/curvature, while moderate regularization allows recovery.


In [ ]:
posteriors, recovered = {}, {}
for scale in candidate_scales:
    posterior = continuum_posterior(obs, model_flux, basis=basis,
                                    sigma_constant=sigma_constant,
                                    sigma_continuum=scale, jitter=jitter_true)
    posteriors[scale] = posterior
    recovered[scale] = np.asarray(evaluate_continuum(basis, posterior.mean))

reference = posteriors[reference_scale]
continuum_variance = np.einsum("ik,kl,il->i", np.asarray(basis),
                              np.asarray(reference.covariance), np.asarray(basis))
continuum_std = np.sqrt(continuum_variance)
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
ax.scatter(wavelength[obs.valid], measured[obs.valid], s=13, color="0.5", label="Usable noisy data")
ax.scatter(wavelength[mask], measured[mask], marker="x", color="0.75", label="Excluded data")
ax.plot(wavelength, true_continuum, "k--", linewidth=2, label="True degree-2 continuum")
for scale, color in zip(candidate_scales, ("tab:orange", "tab:blue", "tab:green")):
    ax.plot(wavelength, recovered[scale], color=color, label=f"Posterior mean; sigma_c={scale:g}")
ax.fill_between(wavelength, recovered[reference_scale]-2*continuum_std,
                recovered[reference_scale]+2*continuum_std, color="tab:blue", alpha=.16,
                label="Conditional ±2 sigma (sigma_c=0.03)")
ax.set(xlabel="Wavelength [Angstrom]", ylabel="Flux / continuum", title="Continuum recovery with Gaussian shrinkage")
ax.legend(fontsize=8, ncol=2)
fig.savefig(work_dir / "continuum-recovery.png", dpi=150)
plt.show()


## B. Regularized overparameterization: five coefficients for degree-2 truth

All five coefficients remain free. Each nonconstant coefficient has the same prior standard deviation; there is no stronger degree-dependent prior. The zero true values of a3/a4 are compared with conditional posterior means and ±2 standard deviations. Compatibility with zero does **not** require exact numerical zeros. The constant coefficient has prior mean 1, while all others have mean 0.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True,
                         gridspec_kw={"width_ratios": [1, 3]})
for offset, (scale, color) in enumerate(zip(candidate_scales, ("tab:orange", "tab:blue", "tab:green"))):
    posterior = posteriors[scale]
    mean = np.asarray(posterior.mean)
    std = np.sqrt(np.diag(np.asarray(posterior.covariance)))
    dx = (offset-1)*.12
    axes[0].errorbar([dx], mean[:1], yerr=2*std[:1], fmt="o", color=color, capsize=3)
    axes[1].errorbar(np.arange(1,5)+dx, mean[1:], yerr=2*std[1:], fmt="o", color=color,
                     capsize=3, label=f"sigma_c={scale:g}")
axes[0].plot([0], true_coefficients[:1], "kD", label="Truth")
axes[0].set(xticks=[0], xticklabels=["a0"], ylabel="Coefficient", title="Free constant")
axes[1].plot(np.arange(1,5), true_coefficients[1:], "kD", label="Truth")
axes[1].axhline(0, color="0.6", linewidth=1)
axes[1].set(xticks=np.arange(1,5), xticklabels=["a1", "a2", "a3", "a4"], title="Shape coefficients (±2 sigma)")
axes[1].legend(fontsize=8)
fig.savefig(work_dir / "continuum-coefficients.png", dpi=150)
plt.show()
print("Reference posterior mean:", np.asarray(reference.mean))
print("a3/a4 distance from zero [posterior sigma]:",
      np.asarray(reference.mean)[3:]/np.sqrt(np.diag(np.asarray(reference.covariance)))[3:])


## C. Marginal likelihoods for prior scale and jitter

Continuum coefficients are integrated out, not optimized or sampled. First vary `sigma_continuum` with fixed `sigma_constant` and known jitter. Then vary jitter with fixed reference prior scale. Both curves include the full Gaussian normalization: effective errors enter **whitening and the determinant**. Increasing jitter can accommodate scatter, but also carries the variance-normalization penalty.

These are separate 1D illustrations, not a joint parameter-estimation workflow. Curve maxima need not equal injected values in a finite noisy realization. A future user-written NumPyro model could assign priors to either hyperparameter; these low-level helpers impose no jitter prior.


In [ ]:
scale_grid = np.logspace(-4, -.3, 80)
scale_logp = np.asarray(jax.jit(jax.vmap(lambda sc:
    marginalized_continuum_log_likelihood(obs, model_flux, basis=basis,
        sigma_constant=sigma_constant, sigma_continuum=sc, jitter=jitter_true)))(jnp.asarray(scale_grid)))
jitter_grid = np.linspace(0., .035, 70)
jitter_logp = np.asarray(jax.jit(jax.vmap(lambda jitter:
    marginalized_continuum_log_likelihood(obs, model_flux, basis=basis,
        sigma_constant=sigma_constant, sigma_continuum=reference_scale, jitter=jitter)))(jnp.asarray(jitter_grid)))
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].semilogx(scale_grid, scale_logp-scale_logp.max(), color="tab:blue")
axes[0].axvline(reference_scale, color="0.5", linestyle="--", label="Reference scale")
axes[0].set(xlabel="Nonconstant coefficient prior scale", ylabel="Marginal log likelihood - maximum",
            title="Gaussian shrinkage hyperparameter")
axes[1].plot(jitter_grid, jitter_logp-jitter_logp.max(), color="tab:purple")
axes[1].axvline(jitter_true, color="0.5", linestyle="--", label="Injected jitter")
axes[1].set(xlabel="Additive jitter [flux units]", ylabel="Marginal log likelihood - maximum",
            title="Residual weighting + normalization")
for ax in axes:
    ax.legend(fontsize=8)
fig.savefig(work_dir / "continuum-hyperparameters.png", dpi=150)
plt.show()
print("Grid maximum sigma_continuum:", scale_grid[np.argmax(scale_logp)])
print("Grid maximum jitter:", jitter_grid[np.argmax(jitter_logp)])
recovery_rmse = np.sqrt(np.mean((recovered[reference_scale]-true_continuum)**2))
print("Recovered continuum RMSE:", recovery_rmse, "; per-pixel noise:", noise_std)
assert recovery_rmse < noise_std
assert np.all(np.isfinite(scale_logp)) and np.all(np.isfinite(jitter_logp))
assert all(p.mean.shape == (5,) and np.all(np.isfinite(p.covariance)) for p in posteriors.values())


The continuum is a data-space multiplicative nuisance model applied after the physical spectrum (including any SB-N composition). Observation and SpecModel retain their existing responsibilities. Independent regions use independent coefficient vectors; this single-region example keeps the mechanism visible. No positivity clipping, per-pixel/fractional jitter, inference framework or external spectral data is introduced.
